## Import libraries

In [1]:
import numpy as np
import pandas as pd
from glob import glob
from obspy import read, UTCDateTime as utc
from datetime import datetime, timezone
from obspy.geodetics.base import gps2dist_azimuth
from tqdm import tqdm

## Load station table

In [11]:
used_station_list=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/station_table/used_stations_100km.csv", sep=",")
used_station_list.head()

,Network,Code,Latitude,Longitude,Elevation,Sensor
0,KS,BAU,35.1027,127.5968,562.0,EL
1,KS,BLGA,36.1392,127.3032,105.0,HG
2,KS,BLLA,34.8646,127.4638,45.0,HG
3,KS,BUYB,36.2726,126.9206,59.0,HH
4,KS,CGPA,34.8768,126.9710,144.0,HG


## Load `PyOcto` output files

### Event table

In [2]:
# event_table_251101=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251101/pyocto_kim1983_prob0.5.csv", sep=",")
event_table_251102=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251102/pyocto_kim1983_prob0.5.csv", sep=",")
# event_table_251103=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251103/pyocto_kim1983_prob0.5.csv", sep=",")
event_table_combined=event_table_251102
# event_table_combined=pd.concat([event_table_251101, event_table_251102, event_table_251103]).reset_index(drop=True)
event_table_combined.head()

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2025-11-02 03:06:07.098809+00:00,5.883866,6.648589,22.421875,9,35.559907,127.314899,22.421875
1,1,2025-11-02 04:13:26.740459+00:00,12.302628,-30.710996,9.296875,14,35.223117,127.385135,9.296875
2,2,2025-11-02 06:20:48.101702+00:00,11.232835,-5.514996,4.921875,8,35.450229,127.373730,4.921875
3,3,2025-11-02 07:36:46.580163+00:00,1.604691,12.730382,30.078125,8,35.614738,127.267712,30.078125
4,4,2025-11-02 10:42:33.640302+00:00,-15.512010,3.173279,0.546875,8,35.528480,127.078970,0.546875


### Pick assignment table

In [3]:
# pick_table_251101=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251101/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
pick_table_251102=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251102/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
# pick_table_251103=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/4.PyOcto/251103/pyocto_assignment_kim1983_prob0.5.csv", sep=",")
pick_table_combined=pick_table_251102
# pick_table_combined=pd.concat([pick_table_251101, pick_table_251102, pick_table_251103]).reset_index(drop=True)
pick_table_combined.head()

,event_idx,pick_idx,residual,station,phase,time
0,0,393,-1.205177,KS.CHOA.,S,1.762053e+09
1,0,790,0.951169,KS.GKSA.,P,1.762053e+09
2,0,1728,-0.514246,KS.GKSA.,S,1.762053e+09
3,0,2521,0.930973,KS.IMSB.,P,1.762053e+09
4,0,2874,0.680056,KS.IMSB.,S,1.762053e+09


In [4]:
pick_table_combined.tail()

,event_idx,pick_idx,residual,station,phase,time
569,38,9139,0.035944,KS.SIDA.,S,1.762122e+09
570,38,9555,0.120055,KS.SUCA.,P,1.762122e+09
571,38,9908,0.264861,KS.SUCA.,S,1.762122e+09
572,38,10510,0.210247,KS.TAIA.,P,1.762122e+09
573,38,10954,0.250285,KS.TAIA.,S,1.762122e+09


## Add P/S pick information to `SAC` header

In [14]:
for i in range(len(pick_table_combined)):
    event_id=str(pick_table_combined["event_idx"][i]+1).zfill(3)
    phase=pick_table_combined["phase"][i]
    pick_time=utc(datetime.fromtimestamp(pick_table_combined["time"][i], tz=timezone.utc))
    network=pick_table_combined["station"][i].split(".")[0]
    station=pick_table_combined["station"][i].split(".")[1]
    channel=used_station_list[used_station_list["Code"]==station].reset_index(drop=True)["Sensor"][0]
    
    sacfile_Z=f"/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/event_waveforms/{event_id}/{event_id}.{network}.{station}..{channel}Z.SAC"
    sacfile_N=f"/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/event_waveforms/{event_id}/{event_id}.{network}.{station}..{channel}N.SAC"
    sacfile_E=f"/home/msseo/works/72.2025_Imsil_swarm/5.Detection_HypoInv/event_waveforms/{event_id}/{event_id}.{network}.{station}..{channel}E.SAC"
    
    tr_Z=read(sacfile_Z)[0]
    tr_N=read(sacfile_N)[0]
    tr_E=read(sacfile_E)[0]
    
    origin_time=tr_Z.stats.starttime-tr_Z.stats.sac.b
    
    if phase=="P":
        tr_Z.stats.sac.a=pick_time-origin_time
        tr_N.stats.sac.a=pick_time-origin_time
        tr_E.stats.sac.a=pick_time-origin_time
    
    elif phase=="S":
        tr_Z.stats.sac.t0=pick_time-origin_time
        tr_N.stats.sac.t0=pick_time-origin_time
        tr_E.stats.sac.t0=pick_time-origin_time
        
    tr_Z.write(sacfile_Z, format="SAC")
    tr_N.write(sacfile_N, format="SAC")
    tr_E.write(sacfile_E, format="SAC")